# Guide 2 — Camera & the YOLO "Eye"

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

Two things the board needs before it can read a card:

1. A **camera** to take a picture of the board.
2. A **YOLO model** — the "eye" that looks at a picture and says *"that's a dog"*
   or *"that's a car."*

All the code here is the real code. It's longer than Guide 1, but you don't have
to memorize it — just understand what each section *does*.


### How this guide fits in

**Depends on:** Guide 1 (needs `MODEL_PATH`, `CLASSES_PATH`). **Used by:** Guides 4, 5.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Step 1: Turn on the YOLO eye

This loads the YOLO model onto the special chip (the DPU) that runs it fast.


In [ ]:
overlay = DpuOverlay('dpu.bit')
overlay.load_model(str(MODEL_PATH))

### Step 2: Load the list of things it can recognize

`voc_classes.txt` is the list of objects YOLO knows (dog, car, person, …).
`YOLO_SCORE_THRESHOLD` is how *sure* YOLO has to be before it says it sees
something. Lower it if it keeps missing cards; raise it if it sees things that
aren't there.


In [ ]:
def get_class(classes_path):
    with open(classes_path) as f:
        return [c.strip() for c in f.readlines()]


class_names = get_class(CLASSES_PATH)

# Lower this if detections are consistently missed; raise it if you get false positives.
YOLO_SCORE_THRESHOLD = 0.2

anchor_list = [10, 13, 16, 30, 33, 23, 30, 61, 62, 45, 59, 119, 116, 90, 156, 198, 373, 326]
anchors = np.array([float(x) for x in anchor_list]).reshape(-1, 2)

num_classes = len(class_names)
hsv_tuples = [(1.0 * x / num_classes, 1.0, 1.0) for x in range(num_classes)]
colors = [tuple(int(c * 255) for c in colorsys.hsv_to_rgb(*hsv)) for hsv in hsv_tuples]
random.seed(0)
random.shuffle(colors)
random.seed(None)

### Step 3: The YOLO math helpers

This is the math that turns YOLO's raw output into neat answers like *"dog, 92%
sure, at this spot."* You do **not** need to understand every line — it's the same
math from the earlier PYNQ 301 lesson. Just run it.


In [ ]:
def letterbox_image(image, size):
    ih, iw, _ = image.shape
    w, h = size
    scale = min(w / iw, h / ih)
    nw, nh = int(iw * scale), int(ih * scale)

    image = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
    new_image = np.ones((h, w, 3), np.uint8) * 128
    h_start, w_start = (h - nh) // 2, (w - nw) // 2
    new_image[h_start:h_start + nh, w_start:w_start + nw, :] = image
    return new_image


def pre_process(image, model_image_size):
    image = image[..., ::-1]
    boxed_image = letterbox_image(image, tuple(reversed(model_image_size)))
    image_data = np.array(boxed_image, dtype='float32') / 255.0
    return np.expand_dims(image_data, 0)


def _get_feats(feats, anchors, num_classes, input_shape):
    num_anchors = len(anchors)
    anchors_tensor = np.reshape(np.array(anchors, dtype=np.float32), [1, 1, 1, num_anchors, 2])
    grid_size = np.shape(feats)[1:3]
    nu = num_classes + 5
    predictions = np.reshape(feats, [-1, grid_size[0], grid_size[1], num_anchors, nu])
    grid_y = np.tile(np.reshape(np.arange(grid_size[0]), [-1, 1, 1, 1]), [1, grid_size[1], 1, 1])
    grid_x = np.tile(np.reshape(np.arange(grid_size[1]), [1, -1, 1, 1]), [grid_size[0], 1, 1, 1])
    grid = np.array(np.concatenate([grid_x, grid_y], axis=-1), dtype=np.float32)

    box_xy = (1 / (1 + np.exp(-predictions[..., :2])) + grid) / np.array(grid_size[::-1], dtype=np.float32)
    box_wh = np.exp(predictions[..., 2:4]) * anchors_tensor / np.array(input_shape[::-1], dtype=np.float32)
    box_confidence = 1 / (1 + np.exp(-predictions[..., 4:5]))
    box_class_probs = 1 / (1 + np.exp(-predictions[..., 5:]))
    return box_xy, box_wh, box_confidence, box_class_probs


def correct_boxes(box_xy, box_wh, input_shape, image_shape):
    box_yx, box_hw = box_xy[..., ::-1], box_wh[..., ::-1]
    input_shape = np.array(input_shape, dtype=np.float32)
    image_shape = np.array(image_shape, dtype=np.float32)
    new_shape = np.around(image_shape * np.min(input_shape / image_shape))
    offset = (input_shape - new_shape) / 2.0 / input_shape
    scale = input_shape / new_shape
    box_yx = (box_yx - offset) * scale
    box_hw *= scale

    box_mins, box_maxes = box_yx - box_hw / 2.0, box_yx + box_hw / 2.0
    boxes = np.concatenate([box_mins[..., 0:1], box_mins[..., 1:2], box_maxes[..., 0:1], box_maxes[..., 1:2]], axis=-1)
    boxes *= np.concatenate([image_shape, image_shape], axis=-1)
    return boxes


def boxes_and_scores(feats, anchors, classes_num, input_shape, image_shape):
    box_xy, box_wh, box_confidence, box_class_probs = _get_feats(feats, anchors, classes_num, input_shape)
    boxes = np.reshape(correct_boxes(box_xy, box_wh, input_shape, image_shape), [-1, 4])
    box_scores = np.reshape(box_confidence * box_class_probs, [-1, classes_num])
    return boxes, box_scores


def nms_boxes(boxes, scores):
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1, yy1 = np.maximum(x1[i], x1[order[1:]]), np.maximum(y1[i], y1[order[1:]])
        xx2, yy2 = np.minimum(x2[i], x2[order[1:]]), np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0.0, xx2 - xx1 + 1) * np.maximum(0.0, yy2 - yy1 + 1)
        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        order = order[np.where(ovr <= 0.55)[0] + 1]
    return keep


def evaluate(yolo_outputs, image_shape, class_names, anchors, score_thresh=None):
    score_thresh = YOLO_SCORE_THRESHOLD if score_thresh is None else score_thresh
    anchor_mask = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
    boxes, box_scores = [], []
    input_shape = np.array(np.shape(yolo_outputs[0])[1:3]) * 32

    for i in range(len(yolo_outputs)):
        b, s = boxes_and_scores(yolo_outputs[i], anchors[anchor_mask[i]], len(class_names), input_shape, image_shape)
        boxes.append(b)
        box_scores.append(s)

    boxes = np.concatenate(boxes, axis=0)
    box_scores = np.concatenate(box_scores, axis=0)
    mask = box_scores >= score_thresh
    boxes_, scores_, classes_ = [], [], []

    for c in range(len(class_names)):
        class_boxes = boxes[mask[:, c]]
        class_scores = box_scores[:, c][mask[:, c]]
        keep = nms_boxes(class_boxes, class_scores)
        boxes_.append(class_boxes[keep])
        scores_.append(class_scores[keep])
        classes_.append(np.ones(len(keep), dtype=np.int32) * c)

    if not boxes_:
        return np.array([]), np.array([]), np.array([])
    return np.concatenate(boxes_, axis=0), np.concatenate(scores_, axis=0), np.concatenate(classes_, axis=0)

### Step 4: The `run` function

This ties it together: give it a picture, and `run(frame)` gives you back what
YOLO found. This is the function the rest of the program calls to "look" at a card.


In [ ]:
dpu = overlay.runner
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()
shapeIn = tuple(inputTensors[0].dims)
shapeOut0, shapeOut1, shapeOut2 = (tuple(t.dims) for t in outputTensors)

input_data = [np.empty(shapeIn, dtype=np.float32, order='C')]
output_data = [
    np.empty(shapeOut0, dtype=np.float32, order='C'),
    np.empty(shapeOut1, dtype=np.float32, order='C'),
    np.empty(shapeOut2, dtype=np.float32, order='C'),
]
image_buffer = input_data[0]


def run(frame, score_thresh=None):
    image_size = frame.shape[:2]
    image_data = np.array(pre_process(frame, (416, 416)), dtype=np.float32)
    image_buffer[0, ...] = image_data.reshape(shapeIn[1:])

    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)

    yolo_outputs = [
        np.reshape(output_data[0], shapeOut0),
        np.reshape(output_data[1], shapeOut1),
        np.reshape(output_data[2], shapeOut2),
    ]
    return evaluate(yolo_outputs, image_size, class_names, anchors, score_thresh=score_thresh)

### Step 5: The camera

This finds the camera and takes fresh pictures. The clever bit: it throws away a
few old frames first, so you always get an up-to-date picture (not the *last*
card). Run it as-is.


In [ ]:
CAMERA_SIZE = (1920, 1080)
CAMERA_FPS = 15
CAMERA_BUFFER_SIZE = 1

# Camera rotation is intentionally not configured manually. The semantic
# TL/TR/BR/BL marker IDs orient every captured board automatically.


def video_devices():
    devices = []
    for name in os.listdir('/dev'):
        if name.startswith('video') and name[5:].isdigit():
            devices.append((int(name[5:]), f'/dev/{name}'))
    return [device for _, device in sorted(devices)]


def open_camera(device):
    camera = cv2.VideoCapture(device, cv2.CAP_V4L2)
    camera.set(cv2.CAP_PROP_BUFFERSIZE, CAMERA_BUFFER_SIZE)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_SIZE[0])
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_SIZE[1])
    camera.set(cv2.CAP_PROP_FPS, CAMERA_FPS)
    return camera


def scan_working_camera():
    for device in video_devices():
        camera = open_camera(device)
        ret, _ = camera.read()
        if camera.isOpened() and ret:
            print(f'Using camera from {device}')
            return camera, device
        camera.release()
    raise RuntimeError('Could not find a working /dev/video* camera.')


def camera_is_open():
    return 'cap' in globals() and cap is not None and cap.isOpened()


def reopen_camera():
    global cap, camera_device
    if camera_is_open():
        cap.release()
    cap, camera_device = scan_working_camera()
    return cap


def ensure_camera_open():
    if not camera_is_open():
        reopen_camera()
    return cap


def drain_camera_buffer(flush_frames=8):
    ensure_camera_open()
    for _ in range(max(0, flush_frames)):
        cap.grab()


def read_camera_frame(read_attempts=8):
    ensure_camera_open()
    for _ in range(max(1, read_attempts)):
        ret, frame = cap.read()
        if ret and frame is not None:
            return frame
        cap.grab()
        ret, frame = cap.retrieve()
        if ret and frame is not None:
            return frame
        time.sleep(0.05)
    return None


def capture_frame(flush_frames=8, settle_seconds=0.3, read_attempts=8):
    if settle_seconds > 0:
        time.sleep(settle_seconds)
    for attempt in range(2):
        drain_camera_buffer(flush_frames if attempt == 0 else 0)
        frame = read_camera_frame(read_attempts=read_attempts)
        if frame is not None:
            return frame
        reopen_camera()
    raise RuntimeError('Could not read a fresh frame from the camera.')


if 'cap' in globals() and cap is not None:
    cap.release()
try:
    cap, camera_device = scan_working_camera()
except RuntimeError as exc:
    print(f'{exc} Continuing without a camera -- see the Simulated Camera Override section below.')
    cap, camera_device = None, None

# These image widgets are rendered once in the Section 10 dashboard. Updating
# .value replaces the picture in place, so camera/debug output never jumps to
# the Camera cell or creates new output areas.
alignment_image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='640px'))
detection_image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='640px'))
processed_crop_image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='416px'))


def camera_alignment_snapshot():
    """Show the full board in marker-defined canonical orientation."""
    frame = capture_frame(flush_frames=1, settle_seconds=0.0)
    marker_count = calibrate_from_border_markers(frame)
    if marker_count < 4:
        raise RuntimeError(f'Only {marker_count}/4 border markers visible; cannot orient the board.')
    oriented = draw_canonical_board(frame)
    _set_image_widget(alignment_image_widget, oriented)
    return oriented

### Check yourself

1. What does `YOLO_SCORE_THRESHOLD` control, in your own words?
2. Why does the camera throw away a few pictures before keeping one?
